In [1]:
import os

In [2]:
%pwd

'c:\\Users\\sagal\\OneDrive\\Desktop\\Let us build\\Linear_regression_01\\research'

In [3]:
os.chdir('..//')

In [4]:
%pwd

'c:\\Users\\sagal\\OneDrive\\Desktop\\Let us build\\Linear_regression_01'

In [5]:
#entity
from dataclasses import dataclass
from pathlib import Path
import pandas as pd
from typing import List, Dict

In [6]:
#entity
@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    ALL_REQUIRED_FILES: List[str]
    all_schema: Dict[str, str]

In [7]:
from Linear_regression_01.constant import *
from Linear_regression_01.utils.common import read_yaml, create_directories

In [9]:
from pathlib import Path

# Ensure your constant paths are defined
CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")
SCHEMA_FILE_PATH = Path("schema.yaml")  # <-- THIS is the missing link!

In [10]:
#configuration manager
#ultimate config manager file for all data validation
class configurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,     # Access to constants
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath) # read all config and params yaml files
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir = config.root_dir,
            STATUS_FILE = config.STATUS_FILE,
            ALL_REQUIRED_FILES = config.ALL_REQUIRED_FILES,
            all_schema = schema
        )

        return data_validation_config


In [11]:
import os
import urllib.request as request
import tarfile #changes on the basis of the file type if zip use import zipfile
from Linear_regression_01.logging import logger
from Linear_regression_01.utils.common import get_size

In [17]:
# validation code ultimate
#components
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    # simple python code that validates all files

    def validate_all_files_exists(self) -> bool:
        try:
            validation_status = None

            all_files = os.listdir(os.path.join("artifacts", "data_ingestion"))

            for file in all_files:
                if file not in self.config.ALL_REQUIRED_FILES:
                    validation_status = False
                    with open(self.config.STATUS_FILE, "w") as f:
                        f.write(f"validation status: {validation_status}")
                
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, "w") as f:
                        f.write(f"validation status: {validation_status}")

            return validation_status
            
        except Exception as e:
            raise e

    #function to validate against the schema     
    def validate_all_columns(self) -> bool:
        try:
            validation_status = True  # Assume true until proven otherwise

            path_to_csv = os.path.join("artifacts","data_ingestion","housing.csv")

            df = pd.read_csv(path_to_csv)
            all_cols = list(df.columns)
            
            # 2. Grab the expected schema columns
            schema_cols = self.config.all_schema.keys()
            
            # 3. Check every column in the dataset against the schema
            for col in all_cols:
                if col not in schema_cols:
                    validation_status = False
                    break
            
            # 4. Append the result to your status tracking file
            with open(self.config.STATUS_FILE, "a") as f:
                f.write(f"Column schema validation status: {validation_status}\n")
                
            return validation_status
            
        except Exception as e:
            raise e

In [ ]:
# 5 Components

#class DataValidation:
#    def __init__(self, config: DataValidationConfig):
 #       self.config = config


  #  def validate_all_columns(self) -> bool:
   #     try:
    #        validation_status = None
            
     #       df = pd.read_csv(self.config.unzip_dir)
      #      all_cols = list(df.columns)
            
       #     if len(all_cols) > 0:
        #        validation_status = True
         #       with open(self.config.status_file, "w") as f:
                    f.write(f"Validation status: {validation_status}")
          #  else:
           #     validation_status = False
            #    with open(self.config.status_file, "w") as f:
             #       f.write(f"Validation status: {validation_status}")
            
           # return validation_status
            
       # except Exception as e:
        #    raise e

In [18]:
# 6 Pipeline
#ultimate pipeline for data validation
try:
    config = configurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    
    #run chek 1
    files_ok = data_validation.validate_all_files_exists()
    
    #run check 2 if file exists
    if files_ok:
        data_validation.validate_all_columns()
        print("Data validation stage completed successfully!")
    else:
        print("Data validation failed: Missing required files.")
        
except Exception as e:
    raise e

[2026-06-10 22:00:47,985: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-10 22:00:47,988: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-10 22:00:47,991: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-10 22:00:47,993: INFO: common: created directory at artifacts]
[2026-06-10 22:00:47,995: INFO: common: created directory at artifacts/data_validation]
Data validation stage completed successfully!
